# Qwen P2 — grid loudness per layer

Which layer is most **grid**-loaded, over the whole training set. The answer should pin
`--select-candidate-layers` (`LAYER`) in `grid/2_dataset_creation/1_p2_selection/`, which currently
inherits 27 from the direction line.

Input is what `wrappers/qwen_analysis/grid/1_loudest_layer/join_loudness_profile.sh` writes: one
row per reasoning token, one `L{layer}` column per layer, each cell `log P(any grid word)` at
that (token, layer), over the pruned Qwen grid vocabulary.

An **empty** cell means the lens did not cover that layer (the jlens outside its fitted
range) — not "no mass here". It becomes `NaN` and is skipped by the mean.

Most grid mass is AXIS (row/column words; see `data/jlens/README.md`), so a peak here is
largely a coordinate-bookkeeping peak.

In [ ]:
# 1. imports and load the joined tables
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

OUT_DIR = Path("/workspace/loudness_evaluation/qwen_p2_grid_layer_profile")
FIG_DIR = OUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# keep_default_na=False: decoded tokens include the literal string "NA" and empty
# strings, which pandas would otherwise turn into missing values.
jlens = pd.read_csv(OUT_DIR / "jlens_tokens.csv", keep_default_na=False, low_memory=False)
logitlens = pd.read_csv(OUT_DIR / "logitlens_tokens.csv", keep_default_na=False, low_memory=False)

print(f"jlens:     {len(jlens):,} token rows, {jlens.trajectory.nunique():,} trajectories")
print(f"logitlens: {len(logitlens):,} token rows, {logitlens.trajectory.nunique():,} trajectories")
jlens.head()

In [ ]:
# 2. aggregate at row level — every token is one observation
def layer_columns(df):
    """The L{layer} columns, in layer order."""
    cols = [c for c in df.columns if c.startswith("L") and c[1:].isdigit()]
    return sorted(cols, key=lambda c: int(c[1:]))


def loudness_matrix(df):
    """Just the layer columns, numeric. Empty cells (layer not covered) become NaN."""
    cols = layer_columns(df)
    return df[cols].apply(pd.to_numeric, errors="coerce")


jlens_mass = loudness_matrix(jlens)
logitlens_mass = loudness_matrix(logitlens)

print("layers:", [int(c[1:]) for c in layer_columns(jlens)])
print(f"jlens rows: {len(jlens_mass):,}   logitlens rows: {len(logitlens_mass):,}")

In [ ]:
# 3. mean grid loudness per layer
def mean_per_layer(mass):
    """{layer: mean loudness}, as a Series indexed by integer layer."""
    means = mass.mean()  # skips NaN
    means.index = [int(c[1:]) for c in means.index]
    return means.sort_index()


jlens_per_layer = mean_per_layer(jlens_mass)
logitlens_per_layer = mean_per_layer(logitlens_mass)

print(f"jlens argmax layer:     {jlens_per_layer.idxmax()}  ({jlens_per_layer.max():.4f})")
print(f"logitlens argmax layer: {logitlens_per_layer.idxmax()}  ({logitlens_per_layer.max():.4f})")
pd.DataFrame({"jlens": jlens_per_layer, "logitlens": logitlens_per_layer})

In [ ]:
# 4. J-Lens Grid Loudness Per Layer
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(jlens_per_layer.index, jlens_per_layer.values, marker="o")
ax.axvline(jlens_per_layer.idxmax(), color="crimson", linestyle="--", linewidth=1)
ax.set_title("J-Lens Grid Loudness Per Layer")
ax.set_xlabel("layer")
ax.set_ylabel("J-lens loudness")
ax.grid(alpha=0.3)
plt.tight_layout()
fig.savefig(FIG_DIR / "jlens_loudness_per_layer.png", dpi=160, bbox_inches="tight")
print("saved", FIG_DIR / "jlens_loudness_per_layer.png")
plt.show()

In [ ]:
# 5. Logitlens Grid Loudness Per Layer
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(logitlens_per_layer.index, logitlens_per_layer.values, marker="o", color="tab:orange")
ax.axvline(logitlens_per_layer.idxmax(), color="crimson", linestyle="--", linewidth=1)
ax.set_title("Logitlens Grid Loudness Per Layer")
ax.set_xlabel("layer")
ax.set_ylabel("Logitlens loudness")
ax.grid(alpha=0.3)
plt.tight_layout()
fig.savefig(FIG_DIR / "logitlens_loudness_per_layer.png", dpi=160, bbox_inches="tight")
print("saved", FIG_DIR / "logitlens_loudness_per_layer.png")
plt.show()